In [0]:
from pyspark.sql.functions import col, lit, when, monotonically_increasing_id, current_timestamp, upper, trim
from pyspark.sql.window import Window
# Accessing your specific catalog and creating the gold schema
spark.sql("USE CATALOG iot_catalog_p2")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
print("✅ Gold schema 'iot_catalog_p2.gold' initialized")

In [0]:
# Create Dimension from confirmed Silver columns
dim_sensor = spark.table("silver.silver_sensor_stream").select(
    "sensor_identifier", 
    "sensor_category"
).distinct() \
 .withColumn("sensor_sk", monotonically_increasing_id()) # Surrogate Key

dim_sensor.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.dim_sensor_master")

print("✅ dim_sensor_master created.")

In [0]:
# Create Dimension from silver_device_operations
dim_device = spark.table("silver.silver_device_operations").select(
    "device_id", 
    "firmware_version", 
    "operation_mode"
).distinct() \
 .withColumn("device_sk", monotonically_increasing_id())

dim_device.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.dim_device_master")

print("✅ dim_device_master created.")

In [0]:
# Create Dimension from silver_environment_network
# We use connection_id as the link to device_id
dim_location = spark.table("silver.silver_environment_network").select(
    "region_name", 
    "network_provider"
).distinct() \
 .withColumn("location_sk", monotonically_increasing_id())

dim_location.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.dim_location_environment")

print("✅ dim_location_environment created.")

In [0]:
# Create Dimension from silver_time_events
dim_time_events = spark.table("silver.silver_time_events").select(
    "event_type"
).distinct() \
 .withColumn("event_sk", monotonically_increasing_id())

dim_time_events.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.dim_time_events")

print("✅ dim_time_events created.")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg, stddev, col, lit, when, monotonically_increasing_id

# 1. Load Silver Metrics
s_stream = spark.table("silver.silver_sensor_stream")
s_ops = spark.table("silver.silver_device_operations")
s_health = spark.table("silver.silver_health_diagnostics")
s_env = spark.table("silver.silver_environment_network")
s_time = spark.table("silver.silver_time_events")

# --- FIXED: Calculate Z-Score handling Division by Zero ---
window_spec = Window.partitionBy("sensor_identifier")
s_stream_with_metrics = s_stream \
    .withColumn("avg_val", avg("sensor_reading_value").over(window_spec)) \
    .withColumn("std_val", stddev("sensor_reading_value").over(window_spec)) \
    .withColumn("z_score",
        when((col("std_val") == 0) | col("std_val").isNull(), lit(0))
        .otherwise((col("sensor_reading_value") - col("avg_val")) / col("std_val")))

# 2. Load Dimensions for Surrogate Keys
d_sensor = spark.table("gold.dim_sensor_master")
d_device = spark.table("gold.dim_device_master")
d_location = spark.table("gold.dim_location_environment")
d_events = spark.table("gold.dim_time_events")

# 3. Build the Fact Table
fact_iot_events = s_stream_with_metrics.alias("s") \
    .join(s_ops.alias("o"), "device_id", "left") \
    .join(s_health.alias("h"), "device_id", "left") \
    .join(s_env.alias("e"), col("s.device_id") == col("e.connection_id"), "left") \
    .join(s_time.alias("t"), col("s.device_id") == col("t.event_id"), "left") \
    .join(d_sensor.alias("ds"), "sensor_identifier", "left") \
    .join(d_device.alias("dd"), "device_id", "left") \
    .join(d_location.alias("dl"), "region_name", "left") \
    .join(d_events.alias("de"), "event_type", "left") \
    .select(
        monotonically_increasing_id().alias("fact_key"),
        col("ds.sensor_sk"),
        col("dd.device_sk"),
        col("dl.location_sk"),
        col("de.event_sk"),
        col("s.event_timestamp"),
        col("s.sensor_reading_value"),
        col("s.z_score"),                # This column is now safe from division by zero
        col("o.cpu_usage_percent"),
        col("h.battery_voltage"),
        col("h.health_indicator")
    )

# 4. Save to Gold
fact_iot_events.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.fact_iot_events")

print("✅ fact_iot_events created successfully with Division-by-Zero protection!")

In [0]:
# 1. Referential Integrity Check
# Every fact should have a valid sensor_sk and device_sk
orphans = spark.sql("""
    SELECT 
        (SELECT COUNT(*) FROM gold.fact_iot_events f 
         LEFT JOIN gold.dim_sensor_master s ON f.sensor_sk = s.sensor_sk 
         WHERE s.sensor_sk IS NULL) as orphaned_sensors,
        (SELECT COUNT(*) FROM gold.fact_iot_events f 
         LEFT JOIN gold.dim_device_master d ON f.device_sk = d.device_sk 
         WHERE d.device_sk IS NULL) as orphaned_devices
""")

display(orphans)

# 2. Anomaly Distribution Check
# Ensure we actually have data to visualize
print("--- Anomaly Summary ---")
spark.table("gold.fact_iot_events") \
    .filter("sensor_reading_value IS NOT NULL") \
    .selectExpr("count(*) as total_readings", "count(if(abs(sensor_reading_value) > 0, 1, null)) as active_readings") \
    .show()

In [0]:
# Load the current fact table
fact_df = spark.table("iot_catalog_p2.gold.fact_iot_events")

# Identify the columns that should uniquely identify a single reading
# We exclude the 'fact_key' because it is unique for every row, even duplicates
business_keys = ["sensor_sk", "device_sk", "event_timestamp", "sensor_reading_value"]

# Deduplicate
clean_fact_df = fact_df.dropDuplicates(business_keys)

# Overwrite the table with the clean version
clean_fact_df.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("iot_catalog_p2.gold.fact_iot_events")

print(f"✅ Deduplication complete. New record count: {clean_fact_df.count()}")